# PP-OCRv5 Phase 1 — Synthetic Pretraining

**Epic:** TTV-119

Train PP-OCRv5 mobile (SVTR_LCNet) on 200K synthetic bib number images.

**Architecture:** MobileNetV1Enhance backbone + SVTR neck + MultiHead (CTC+SAR)

**Charset:** 0-9 only (digits), max_text_length=4

---

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# Install PaddlePaddle + PaddleOCR
!pip install -q paddlepaddle-gpu==2.6.2
!pip install -q paddleocr lmdb
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
!cd /content/PaddleOCR && pip install -q -r requirements.txt

import paddle
print(f'PaddlePaddle: {paddle.__version__}')
print(f'GPU: {paddle.device.is_compiled_with_cuda()}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/cycling-photo-ai/experiments'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

## 1. Upload synthetic data

Upload the synthetic LMDB from local to Colab. Two options:

**Option A:** Upload `data/ocr/synthetic/lmdb/` to Drive, then copy here

**Option B:** Generate fresh on Colab (takes ~10 min)

We'll use Option A — copy from Drive:

In [ ]:
import shutil
from pathlib import Path

# Check if synthetic data is on Drive
DRIVE_SYNTH = Path('/content/drive/MyDrive/cycling-photo-ai/data/ocr/synthetic/lmdb')

if DRIVE_SYNTH.exists():
    print('Synthetic LMDB found on Drive')
else:
    print('Synthetic LMDB not on Drive.')
    print('Upload data/ocr/synthetic/lmdb/ to Drive at:')
    print(f'  {DRIVE_SYNTH}')
    print('Or run the generation cell below.')

In [ ]:
# Option B: Generate synthetic data directly on Colab
# Only run this if you didn't upload LMDB to Drive

GENERATE_FRESH = not DRIVE_SYNTH.exists()  # auto-detect

if GENERATE_FRESH:
    !pip install -q trdg
    
    # Copy generator script from repo
    # You may need to upload generate_synthetic_bibs.py manually
    print('TODO: upload and run generate_synthetic_bibs.py')
    print('Or upload LMDB to Drive and re-run previous cell')

## 2. Convert LMDB to PaddleOCR format

In [ ]:
import io
import lmdb
from PIL import Image

SYNTH_LMDB = str(DRIVE_SYNTH) if DRIVE_SYNTH.exists() else '/content/synthetic/lmdb'
PPOCR_DIR = Path('/content/ppocr_data')
IMGS_DIR = PPOCR_DIR / 'images'
IMGS_DIR.mkdir(parents=True, exist_ok=True)

env = lmdb.open(SYNTH_LMDB, readonly=True, lock=False)
with env.begin() as txn:
    n = int(txn.get('num-samples'.encode()).decode())
print(f'Total samples: {n}')

n_train = int(n * 0.95)
train_labels = []
val_labels = []

with env.begin() as txn:
    for idx in range(n):
        img_bytes = txn.get(f'image-{idx+1:09d}'.encode())
        label = txn.get(f'label-{idx+1:09d}'.encode()).decode()
        
        fname = f'syn_{idx:07d}.jpg'
        img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        img = img.resize((192, 48), Image.LANCZOS)
        img.save(IMGS_DIR / fname, quality=95)
        
        entry = f'images/{fname}\t{label}'
        if idx < n_train:
            train_labels.append(entry)
        else:
            val_labels.append(entry)
        
        if (idx + 1) % 50000 == 0:
            print(f'  {idx+1}/{n}')

env.close()

(PPOCR_DIR / 'train_label.txt').write_text('\n'.join(train_labels))
(PPOCR_DIR / 'val_label.txt').write_text('\n'.join(val_labels))

# Digit dictionary
(PPOCR_DIR / 'digits_dict.txt').write_text('\n'.join('0123456789'))

print(f'Train: {len(train_labels)}, Val: {len(val_labels)}')

## 3. Create training config

In [ ]:
OUTPUT_DIR = '/content/ppocr_output'

config_content = f"""Global:
  debug: false
  use_gpu: true
  epoch_num: 50
  log_smooth_window: 20
  print_batch_step: 100
  save_model_dir: {OUTPUT_DIR}
  save_epoch_step: 10
  eval_batch_step: [0, 1000]
  cal_metric_during_train: true
  checkpoints: null
  pretrained_model: null
  save_inference_dir: null
  use_visualdl: false
  character_dict_path: {str(PPOCR_DIR / 'digits_dict.txt')}
  max_text_length: 4
  infer_mode: false
  use_space_char: false
  distributed: false

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.001
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 0.00001

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform: null
  Backbone:
    name: MobileNetV1Enhance
    scale: 0.5
    last_conv_stride: [1, 2]
    last_pool_type: avg
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 64
            depth: 2
            hidden_dims: 120
            use_guide: true
          Head:
            fc_decay: 0.00001
      - SARHead:
          enc_dim: 512
          max_text_length: 4

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss: null
    - SARLoss: null

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc
  ignore_space: false

Train:
  dataset:
    name: SimpleDataSet
    data_dir: {str(PPOCR_DIR)}
    label_file_list:
      - {str(PPOCR_DIR / 'train_label.txt')}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - RecAug: null
      - MultiLabelEncode: null
      - RecResizeImg:
          image_shape: [3, 48, 192]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_sar
            - length
            - valid_ratio
  loader:
    shuffle: true
    batch_size_per_card: 128
    drop_last: true
    num_workers: 4

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: {str(PPOCR_DIR)}
    label_file_list:
      - {str(PPOCR_DIR / 'val_label.txt')}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode: null
      - RecResizeImg:
          image_shape: [3, 48, 192]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_sar
            - length
            - valid_ratio
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 128
    num_workers: 2
"""

config_path = '/content/ppocr_config.yml'
with open(config_path, 'w') as f:
    f.write(config_content)
print(f'Config written to {config_path}')

## 4. Train

In [ ]:
import time
start = time.time()

!cd /content/PaddleOCR && python tools/train.py -c /content/ppocr_config.yml

elapsed = (time.time() - start) / 60
print(f'\nTraining time: {elapsed:.1f} minutes')

## 5. Evaluate + save results

In [ ]:
# Find best model
import glob
best_model = None
for f in glob.glob(f'{OUTPUT_DIR}/best_accuracy*'):
    best_model = f.replace('.pdparams', '').replace('.pdopt', '').replace('.states', '')
    break

if best_model:
    print(f'Best model: {best_model}')
    !cd /content/PaddleOCR && python tools/eval.py \
        -c /content/ppocr_config.yml \
        -o Global.checkpoints={best_model}
else:
    print('No best model found — check training logs')
    !ls -la {OUTPUT_DIR}/

In [ ]:
# Save to Drive
import shutil

drive_run_dir = Path(DRIVE_OUTPUT) / 'ocr_phase1_ppocr'
if drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)

shutil.copytree(OUTPUT_DIR, drive_run_dir)
print(f'Saved to {drive_run_dir}')

# List saved files
for f in sorted(drive_run_dir.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.relative_to(drive_run_dir)} ({size_mb:.1f} MB)')

In [ ]:
print('='*60)
print('RESUMEN PARA EXPERIMENT_LOG_OCR.md')
print('='*60)
print(f'\n### Run 2 — PP-OCRv5 Phase 1 (Synthetic)')
print(f'- **Fecha:** {time.strftime("%Y-%m-%d")}')
print(f'- **GPU:** {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'- **Training time:** {elapsed:.1f} min')
print(f'- **Architecture:** SVTR_LCNet (MobileNetV1Enhance + SVTR neck + CTC+SAR head)')
print(f'- **Dataset:** 200K synthetic (190K train / 10K val)')
print(f'\nCheck training logs above for best accuracy.')